In [1]:
import venti
import numpy as np

In [2]:
print(dir(venti))

['__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_version', 'models']


In [3]:
from venti.models import plate_motion

In [4]:
# Your GPS data
longitude = np.array([-120.0, -115.0, -110.0, -105.0, -100.0])
latitude = np.array([35.0, 40.0, 45.0, 40.0, 35.0])
velocity_east = np.array([2.1, 1.8, 1.5, 1.2, 0.9])  # mm/year
velocity_north = np.array([0.5, 0.8, 1.1, 1.4, 1.7])  # mm/year
sigma_east = np.array([0.1, 0.15, 0.12, 0.11, 0.13])  # mm/year
sigma_north = np.array([0.12, 0.14, 0.11, 0.13, 0.15])  # mm/year

# Calculate Euler pole
euler_lon, euler_lat, omega, stats = plate_motion.calculate_euler_pole(
    longitude, latitude, velocity_east, velocity_north,
    sigma_east, sigma_north
)

print(f"Euler Pole: {euler_lon:.2f}°, {euler_lat:.2f}°")
print(f"Angular velocity: {omega:.3f} °/Myr")
print(f"RMS: {stats['rms']:.2f} mm/year")


Euler Pole: -136.47°, 57.02°
Angular velocity: 0.039 °/Myr
RMS: 0.59 mm/year


In [5]:
max_sigma, min_sigma, azimuth, sigma_omega = plate_motion.get_euler_pole_uncertainty(
            euler_lon, euler_lat, omega, stats['parameter_covariance'])

In [6]:
# Display results
print(f"\nEULER POLE RESULTS:")
print(f"  Longitude: {euler_lon:.3f}° ± {max_sigma:.3f}°")
print(f"  Latitude:  {euler_lat:.3f}° ± {min_sigma:.3f}°")
print(f"  Rotation:  {omega:.4f} ± {sigma_omega:.4f} °/Myr")

print(f"\nUNCERTAINTY ELLIPSE:")
print(f"  Semi-major axis: {max_sigma:.3f}°")
print(f"  Semi-minor axis: {min_sigma:.3f}°")
print(f"  Ellipse azimuth: {azimuth:.1f}°")
print(f"  Ellipticity: {max_sigma/min_sigma:.2f}")

print(f"\nFIT QUALITY STATISTICS:")
print(f"  RMS:              {stats['rms']:.2f} mm/year")
print(f"  Weighted RMS:     {stats['wrms']:.2f}")
print(f"  Chi-squared:      {stats['chi_squared']:.2f}")
print(f"  Reduced chi-sq:   {stats['reduced_chi_squared']:.2f}")
print(f"  Degrees of freedom: {stats['degrees_of_freedom']}")

print(f"\nINTERPRETATION:")
if stats['reduced_chi_squared'] < 1.5:
    print(f"  Good fit: reduced χ² = {stats['reduced_chi_squared']:.2f} < 1.5")
elif stats['reduced_chi_squared'] < 3.0:
    print(f"  Acceptable fit: reduced χ² = {stats['reduced_chi_squared']:.2f}")
else:
    print(f"  Poor fit: reduced χ² = {stats['reduced_chi_squared']:.2f} > 3.0")

if max_sigma < 5.0:
    print(f"  Good position uncertainty: {max_sigma:.2f}° < 5°")
elif max_sigma < 10.0:
    print(f"  Moderate position uncertainty: {max_sigma:.2f}°")
else:
    print(f"  Large position uncertainty: {max_sigma:.2f}° > 10°")

if sigma_omega / omega < 0.1:
    print(f"  Good rotation rate uncertainty: {sigma_omega/omega:.1%} < 10%")
elif sigma_omega / omega < 0.2:
    print(f"  Moderate rotation rate uncertainty: {sigma_omega/omega:.1%}")
else:
    print(f"  Large rotation rate uncertainty: {sigma_omega/omega:.1%} > 20%")


EULER POLE RESULTS:
  Longitude: -136.471° ± 2.768°
  Latitude:  57.016° ± 0.742°
  Rotation:  0.0390 ± 0.0040 °/Myr

UNCERTAINTY ELLIPSE:
  Semi-major axis: 2.768°
  Semi-minor axis: 0.742°
  Ellipse azimuth: -36.1°
  Ellipticity: 3.73

FIT QUALITY STATISTICS:
  RMS:              0.59 mm/year
  Weighted RMS:     0.41
  Chi-squared:      110.80
  Reduced chi-sq:   3.98
  Degrees of freedom: 7

INTERPRETATION:
  Poor fit: reduced χ² = 3.98 > 3.0
  Good position uncertainty: 2.77° < 5°
  Moderate rotation rate uncertainty: 10.1%


In [7]:
modeled_vel = plate_motion.model_velocities_from_euler_pole(longitude, latitude, -107.2, 50.8, 0.65)
modeled_vel

array([[ 20.54729342, -10.1207296 ],
       [ 14.0448572 ,  -6.19797774],
       [  7.57972192,  -2.23026388],
       [ 13.79587853,   1.75312812],
       [ 20.10476875,   5.72544513]])

In [8]:
modeled_vel, uncertainties = plate_motion.model_velocities_from_euler_pole(
    longitude, latitude, euler_lon, euler_lat, omega,
    euler_covariance=stats['parameter_covariance']  # From calculate_euler_pole()
)
modeled_vel, uncertainties

(array([[1.6919189 , 0.66871286],
        [1.38586585, 0.86304367],
        [1.08976059, 1.05068874],
        [1.50316098, 1.23094777],
        [1.90039455, 1.40192818]]),
 array([[0.06377453, 0.08307369],
        [0.05391554, 0.06363358],
        [0.07385925, 0.05751117],
        [0.05382035, 0.06856027],
        [0.0643157 , 0.0904695 ]]))

In [9]:
itrf14_pmm = plate_motion.json_to_dataframe(plate_motion.load_itrf_json(date=2014))
itrf20_pmm = plate_motion.json_to_dataframe(plate_motion.load_itrf_json(date=2020))

In [20]:
import pandas as pd
itrf14_gps = pd.read_csv(plate_motion.ITRF14_DATA)
itrf20_gps = pd.read_csv(plate_motion.ITRF20_DATA)

In [21]:
pacific_data = itrf14_gps[itrf14_gps['Plate'] == 'Pacific']

# Calculate Euler pole
longitude = pacific_data['Longitude_deg'].values
latitude = pacific_data['Latitude_deg'].values
ve = pacific_data['Ve_mm_yr'].values
vn = pacific_data['Vn_mm_yr'].values
se = pacific_data['Se_mm_yr'].values
sn = pacific_data['Sn_mm_yr'].values

euler_lon, euler_lat, omega, stats = plate_motion.calculate_euler_pole(
    longitude, latitude, ve, vn, se, sn, correlation_coefficient=0.35, 
)

# Calculate uncertainties
max_sigma, min_sigma, azimuth, sigma_omega = plate_motion.get_euler_pole_uncertainty(
    euler_lon, euler_lat, omega, stats['parameter_covariance']
)

print(f"\nCalculated Pacific Plate Euler Pole:")
print(f"  Longitude: {euler_lon:.3f}°")
print(f"  Latitude:  {euler_lat:.3f}°")
print(f"  Angular velocity: {omega:.4f} °/Myr")
print(f"  RMS: {stats['rms']:.2f} mm/year")

np.rad2deg(stats['rotation_rates_rad_per_year'])*1e6


Calculated Pacific Plate Euler Pole:
  Longitude: 111.197°
  Latitude:  -62.524°
  Angular velocity: 0.6796 °/Myr
  RMS: 0.52 mm/year


array([-0.11336805,  0.29232019, -0.60291383])

In [22]:
itrf14_pmm[itrf14_pmm.name == 'Pacific Plate'] 

,plate,name,omega_x,omega_y,omega_z
8,PCFC,Pacific Plate,-0.1135,0.2907,-0.6025


In [13]:
pacific_data

,CODE,DOMES_Number,Site_Name,Plate,Technique,Longitude_deg,Latitude_deg,Ve_mm_yr,Vn_mm_yr,Se_mm_yr,Sn_mm_yr,Re_mm_yr,Rn_mm_yr
246,POHN,51601M001,Pohnpei,Pacific,P,158.210,6.960,-70.04,25.43,0.012,0.010,0.606,-0.065
247,NAUR,50701M001,Nauru,Pacific,P,166.926,-0.552,-66.96,29.74,0.017,0.015,0.081,-1.072
248,KWJ1,50506M001,Kwajalein atoll,Pacific,PR,167.730,8.722,-69.05,29.60,0.020,0.016,-0.140,-0.659
249,MAJU,51501M001,Majuro,Pacific,P,171.365,7.119,-68.69,30.36,0.029,0.024,0.004,-0.259
250,KIRI,50305M001,Betio island - k,Pacific,P,172.923,1.355,-67.83,31.17,0.010,0.008,0.389,-0.611
251,CHTI,50242M001,Chatham island n,Pacific,D,183.383,-43.735,-40.92,32.72,0.008,0.008,-0.258,0.279
252,CHAT,50207M001,Chatham island,Pacific,DP,183.434,-43.956,-40.72,32.97,0.007,0.006,-0.271,0.038
253,CKIS,50213M003,Rarotonga,Pacific,P,200.199,-21.201,-62.85,35.16,0.008,0.007,0.545,-0.442
254,KOK1,49896S001,Kokole point,Pacific,P,200.242,21.983,-62.30,35.43,0.013,0.011,-0.162,-0.712
255,KOKB,40424M004,Kauai,Pacific,DPR,200.335,22.126,-62.21,34.62,0.007,0.006,-0.185,0.110


In [18]:
pacific_data = itrf20_gps[itrf20_gps['Plate'] == 'Pacific']

# Calculate Euler pole
longitude = pacific_data['Longitude_deg'].values
latitude = pacific_data['Latitude_deg'].values
ve = pacific_data['Ve_mm_yr'].values
vn = pacific_data['Vn_mm_yr'].values
se = pacific_data['Se_mm_yr'].values
sn = pacific_data['Sn_mm_yr'].values

euler_lon, euler_lat, omega, stats = plate_motion.calculate_euler_pole(
    longitude, latitude, ve, vn, se, sn, correlation_coefficient=0.35, 
)

# Calculate uncertainties
max_sigma, min_sigma, azimuth, sigma_omega = plate_motion.get_euler_pole_uncertainty(
    euler_lon, euler_lat, omega, stats['parameter_covariance']
)

print(f"\nCalculated Pacific Plate Euler Pole:")
print(f"  Longitude: {euler_lon:.3f}°")
print(f"  Latitude:  {euler_lat:.3f}°")
print(f"  Angular velocity: {omega:.4f} °/Myr")
print(f"  RMS: {stats['rms']:.2f} mm/year")

np.rad2deg(stats['rotation_rates_rad_per_year'])*1e6


Calculated Pacific Plate Euler Pole:
  Longitude: 111.396°
  Latitude:  -62.585°
  Angular velocity: 0.6778 °/Myr
  RMS: 0.53 mm/year


array([-0.11385903,  0.29058895, -0.60171068])

In [26]:
stats['rotation_rates_rad_per_year']

[np.float64(-1.978645702459456e-09),
 np.float64(5.1019498158727684e-09),
 np.float64(-1.0522831413805372e-08)]

In [15]:
itrf20_pmm[itrf20_pmm.name == 'Pacific Plate'] 

,plate,name,omega_x,omega_y,omega_z
10,PCFC,Pacific Plate,-0.1122,0.2836,-0.5984


In [25]:
(np.rad2deg(-0.404 * np.pi / 648000000)*1e6,
np.rad2deg(1.021 * np.pi / 648000000)*1e6,
np.rad2deg(-2.154 * np.pi / 648000000)*1e6)

(np.float64(-0.11222222222222224),
 np.float64(0.28361111111111115),
 np.float64(-0.5983333333333334))